# Feedback-controlled adaptive smoothing for OCT defect segmentation

**Idea.** Smoothing removes speckle but blurs boundaries and erases small defects. Instead of one global
smoothing level, a per-pixel strength map **λ ∈ [0, 1]** is chosen *by feedback from the segmenter*:

```
x ──► S(x, λₜ) ──► U-Net + hyperspherical head ──► mask, confidence, boundary ──► controller ──► λₜ₊₁   (T = 3)
```

* **Smoother S (denoising):** λ moves along a scale space `[raw, D(x), D(x)*G(σ=1), D(x)*G(σ=2)]`, where D is a
  DnCNN-style denoiser trained self-supervised with Noise2Void (no clean images needed). λ = 0 means no smoothing; 1 means strong.
* **Sensor (hyperspherical latent space):** per-pixel embeddings on the unit sphere are compared with learnable
  class prototypes (cosine logits) and a *boundary* prototype. The cosine to the nearest class prototype is the
  confidence; the boundary score separates "unsure because it's an edge" from "unsure because it's noisy".
* **Controller:** a small CNN reads (confidence, boundary score, image gradient, λₜ) and updates λ.

**Methods compared**

| method | smoothing | head |
|---|---|---|
| `unet_linear` | none (plain U-Net baseline) | softmax |
| `unet` | none | hyperspherical |
| `fixed_0.33` / `fixed_0.67` / `fixed_1.0` | constant λ (denoiser / +σ1 blur / +σ2 blur) | hyperspherical |
| `open_loop` | λ predicted from the image only (no feedback) | hyperspherical |
| **`feedback`** | **closed-loop λ, 3 iterations (proposed)** | hyperspherical |
| `feedback_linear` | closed-loop λ | softmax (ablation: does the hypersphere matter?) |

**Evaluation** (subject-level 5-fold cross-validation, each subject tested once): Dice/IoU, **boundary F1** (2 px
tolerance), **HD95**, **small-defect recall**, false-positive components, all repeated at extra test-time speckle
σ ∈ {0, 0.1, 0.2, 0.3}; λ statistics by region (boundary / defect interior / background) and Dice per feedback
iteration; on Duke DME also *expert 2 vs expert 1* as a human-agreement ceiling.

At the end the notebook prints `summary.json`; send that file (and `report.md`) back for discussion.

In [ ]:
GPU = 0   # GPU index as listed by nvidia-smi (e.g. 0 or 1), or "cpu". Restart the kernel after changing it.

import os, sys, json, subprocess
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "fbseg").is_dir() else Path.cwd().parent   # works from notebooks/ or the repo root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from nadycnn.devices import describe_device, select_device
select_device(GPU)   # must run before torch first uses CUDA

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image as ShowImage, Markdown, display

from fbseg import DATA_ROOT, RUNS_ROOT
from fbseg.data import SegData
from fbseg.experiment import evaluate_saved, run
from fbseg.models import METHODS, FeedbackSegmenter, count_parameters
from fbseg.plots import (load_runs, plot_lambda_regions, plot_robustness, plot_samples, plot_scale_space,
                         plot_training_curves, save_all_figures)
from fbseg.training import get_device

print("project root:", ROOT, "| device:", describe_device(), "| torch", torch.__version__)

## 1. Data

`scripts/download_data.py` (repo root) writes every dataset to `data/seg/<name>/{images,masks}` (and `masks_expert2` for Duke).

* `duke_dme` (default): Duke DME, Chiu et al. 2015. 10 subjects × 11 annotated Spectralis B-scans with fluid masks
  from 2 graders, ~178 MB, no registration. The download is resumable (the server is slow).
* `synthetic`: OCT-like scans with fluid pockets, generated locally; use it if the download fails.
* `retouch`, `aroi`: require registration; the script prints the manual steps and the expected layout.

In [ ]:
DATASET = "duke_dme"      # "duke_dme" | "synthetic" | any folder you placed under data/seg/
AUTO_DOWNLOAD = True      # run scripts/download_data.py if the dataset folder is missing
FALLBACK_TO_SYNTHETIC = True

if not (Path(DATA_ROOT) / DATASET / "images").is_dir() and AUTO_DOWNLOAD:
    try:
        subprocess.run([sys.executable, "scripts/download_data.py", DATASET], check=True)
    except subprocess.CalledProcessError as error:
        if not FALLBACK_TO_SYNTHETIC:
            raise
        print(f"download failed ({error}); falling back to synthetic data")
        DATASET = "synthetic"
        subprocess.run([sys.executable, "scripts/download_data.py", "synthetic"], check=True)

data = SegData.load(DATASET, Path(DATA_ROOT))
has_defect = (data.masks.sum(dim=(1, 2, 3)) > 0).float().mean().item()
print(f"{DATASET}: {len(data.names)} B-scans, {len(np.unique(data.subjects))} subjects, image {tuple(data.images.shape[2:])}, "
      f"{100 * has_defect:.0f}% contain a defect, defect pixels {100 * data.masks.mean().item():.2f}%, "
      f"second annotator: {data.masks_expert2 is not None}")

In [ ]:
# Three scans with the most defect pixels: image, mask, and the boundary band used for boundary metrics/analysis.
plot_samples(data); plt.show()

## 2. Train (small budget)

Defaults: 30 epochs max (early stopping on validation Dice, patience 15), batch 4, 256×384 input,
5 folds × 1 seed × 8 methods = 40 short runs. Each fold trains on 6 subjects, validates on 2 and tests on 2.
Rough time on a 4 GB laptop GPU: 1–2 min per baseline run and 3–4 min per feedback run, **~1 h total**.
For a first look use `FOLDS = [0]` (~10 min). Finished runs are skipped when you re-run the cell, so you can stop and resume.

Set `RUN_TRAINING = False` to skip training, e.g. when the checkpoints were trained on the server
(`scripts/miniproject_train.py`) and `runs_seg/` was copied back; section 3 then evaluates them.

In [ ]:
RUN_TRAINING = True
METHOD_NAMES = list(METHODS)   # or e.g. ["unet", "fixed_0.67", "open_loop", "feedback"]
FOLDS = None                   # None = all 5 folds; [0] = quick single fold
SEEDS = [0]
EPOCHS = 30

print(f"feedback model parameters: {count_parameters(FeedbackSegmenter(METHODS['feedback'])):,}")
if RUN_TRAINING:
    run(DATASET, METHOD_NAMES, SEEDS, EPOCHS, folds=FOLDS)

## 3. Evaluate the trained models

Loads every checkpoint in `runs_seg/<dataset>/<method>/fold*_seed*/best.pt` (no training) and scores it on its
fold's held-out subjects at each extra speckle level: Dice, IoU, boundary F1, HD95, small/large-defect recall,
false-positive components, λ by region, and Dice per feedback iteration. Then it rebuilds `report.md`,
`summary.json` and the figures in `runs_seg/<dataset>/figures/`. Same as `scripts/miniproject_evaluate.py`.

In [ ]:
EVAL_NOISE_LEVELS = (0.0, 0.1, 0.2, 0.3)   # extra multiplicative speckle sigma at test time
EVAL_METHODS = None                         # None = every method with a checkpoint

report_md, summary = evaluate_saved(DATASET, EVAL_METHODS, EVAL_NOISE_LEVELS)
figure_paths = save_all_figures(Path(RUNS_ROOT), DATASET, data)

## 4. Results

In [ ]:
display(Markdown(report_md))

In [ ]:
runs = load_runs(Path(RUNS_ROOT), DATASET)
plot_robustness(runs, ("dice", "boundary_f1", "small_defect_recall")); plt.show()
plot_robustness(runs, ("hd95",)); plt.show()

In [ ]:
if any(m in runs for m in ("open_loop", "feedback", "feedback_linear")):
    for level in ("sigma_0.0", "sigma_0.3"):
        plot_lambda_regions(runs, level); plt.show()
plot_training_curves(Path(RUNS_ROOT), DATASET); plt.show()

### Qualitative: where does the controller smooth?

Columns: noisy input (extra σ = 0.1), λ after the first and last iteration (bright = more smoothing), the smoothed
scan seen by the segmenter, prediction, ground truth. If the idea works, λ should be dark along fluid boundaries
and small pockets and bright in flat speckled tissue.

In [ ]:
for method in ("feedback", "open_loop", "feedback_linear"):
    path = Path(RUNS_ROOT) / DATASET / method / "fold0_seed0" / "qualitative.png"
    if path.exists():
        print(method); display(ShowImage(filename=str(path)))

In [ ]:
# The trained smoother's scale space: lambda = 0, 1/3, 2/3, 1.
figure = plot_scale_space(Path(RUNS_ROOT), DATASET, data)
if figure is not None:
    plt.show()

## 5. Send these back

`runs_seg/<dataset>/summary.json` (compact numbers) and `runs_seg/<dataset>/report.md` (tables). The cell below
prints both paths and the JSON so you can copy it directly.

`scripts/collect_results.py` zips every report, metrics file and figure (no checkpoints) into `results_bundle/`.

In [ ]:
summary_path = Path(RUNS_ROOT) / DATASET / "summary.json"
print(summary_path.resolve()); print((Path(RUNS_ROOT) / DATASET / "report.md").resolve())
print(summary_path.read_text())